In [ ]:
###!/usr/bin/env python
################################################
# New style 
# ###############################################
import sys

rootdir_ = '../'
if ( rootdir_ not in sys.path ):
    sys.path.append(rootdir_)
    print( f" a path to {rootdir_} added in {__name__} ")


from Utils import GridUtils as GrU
from Utils import MakePressures as MkP
from Utils import utils as uti
from Utils import MyConstants as Co
from Utils import time_utils as tuti
from Utils import numerical_utils as nuti

import analysis_utils as auti
import file_utils as futi
import event_utils as euti


#from PyRegridding.Utils import MakePressures as MkP
#from Drivers import RegridField as RgF
import RegridField as RgF

# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# for smoothing , nonlienar colors ...
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter
import matplotlib.colors as mcolors

# Some other useful packages 
import copy
import time
import cftime
import yaml
import numbers

# Some other useful packages 
import importlib
from pathlib import Path


importlib.reload( auti )
importlib.reload( futi )
importlib.reload( euti )

Rdair=Co.Rdair()


 a path to ../ added in __main__ 


In [ ]:
# This allow both dict.key and dict['key'] syntax
class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")



In [ ]:
%%time
importlib.reload( futi )
importlib.reload( nuti )
nsteps=None
start_date=None
#super_lat_range = [-90.,90.]  #[-85,-30]
super_lat_range = [-80.,-30.]  #[-85,-30]
#super_lat_range = [30.,80.] # Northern Summer!!!!!!!!!!
#case, process_ncdata, start_date, nsteps = 'c153_topfix_ne240pg3_FMTHIST_xic_x02'   , False #, [2004,7,15,0], 248
#case, process_ncdata, start_date, nsteps = 'c153_topfix_ne240pg3_FMTHIST_xic_x03'   , False #, [2007,7,15,0], 248
#case, process_ncdata, start_date, nsteps = 'c153_topfix_ne240pg3_FMTHIST_xic_x03'   , False #, [2007,8,15,0], 124
case, process_ncdata  = 'cam77_dyamond1_prod1'    , False
#case, process_ncdata  = 'c124_dyamond1_prod2'    , False
#case , process_ncdata = 'xy-rdg-mm-front'    , True
#nsteps=8
A = futi.read_case( case=case, nsteps=nsteps, start_date=start_date , super_lat_range=super_lat_range ) # , nsteps = 31*8 )

time, zlev, lat, lon = A.time, A.zlev, A.lat, A.lon



In [ ]:
%%time
#################################################################
# Make event lists ... and composites
importlib.reload(euti)
importlib.reload(auti)

zlev_event=15_000. #23_000.
lat_range=  [-50,-40] #[-70,-60] #[-60,-40]
lat_range=  [-60,-40] #[-70,-60] #[-60,-40]
#lat_range=  [35,65] # Northern Summer!!!!!!!!!!
lon_range=[0,360] # [0,60]
exclude_orography=True

fracs=[0.995,0.90,0.50,0.25,0.125,0.0625]
#fracs=[0.95,0.90,0.50,0.25,0.125,0.0625]

El = euti.make_El(
            A=A, 
            fractions_for_thresholds=fracs,
            zlev_event=zlev_event, 
            lat_range=lat_range,
            lon_range=lon_range,
            exclude_orography= exclude_orography,
            peak_footprint=(3,3),
            return_after_stage1=False
           )




In [ ]:
print(A.epwp.shape)
print( A.epwp.shape[0] * A.epwp.shape[2] * A.epwp.shape[3] )


In [ ]:
# This cell was added on May 7 2026. Trying to make things more efficient by writing 
# the event structures to netcdf files ... Claude wrote code .. includes a 'json sidecar'
# whatever that means ... 
# Going to skip this by default for now
load_El_from_files = False
if load_El_from_files == True:
    import event_io as eio
    importlib.reload(eio)
    
    
    # save
    #paths = eio.save_El(El, prefix='dyamond_SO', outdir='/glade/derecho/scratch/juliob/')
    
    # load
    El, meta = eio.load_El(prefix='dyamond_SO', indir='/glade/derecho/scratch/juliob/May08v5/')
    eio.summarise_El(El, meta)

In [ ]:
importlib.reload( auti )
auti.plot_xavg_compos( fld='zeta_4D', El=El )
auti.plot_xavg_compos( fld='tilt_4D', El=El )
auti.plot_xavg_compos( fld='fgf_4D', El=El )


In [ ]:

ds=El[0].ds
print(int(ds.itime.values.max()) + 1)

ds_drop=ds.drop_vars( ['lat','lon','zlev'] )


event_list=euti.ds_to_event_list( ds_drop )

In [ ]:
evoo=event_list[120]
len(evoo)
np.array(evoo[1]['ix'])

In [ ]:
import random_forest as RF

Eco=   El[0]  #euti.combine_event_dicts(El[3], El[1], label_key='event_strength')

In [ ]:
Eco.zeta_4D.shape

In [ ]:
#### SAVE OFF events dataset
_t=0
euti.write_ds(ds=El[_t].ds, A=A,fraction_of_total=El[_t].Frac_of_total_epwp,
            zlev_event=zlev_event,thresh=El[_t].threshold,
            lat_range=lat_range,lon_range=lon_range,extra_info='_NoTopo' )


In [ ]:
from scipy import stats

#zlev=Eco.zlev

z0=np.argmin( np.abs( zlev-0.))
z3=np.argmin( np.abs( zlev-3000.))
z5=np.argmin( np.abs( zlev-5000.))
z6=np.argmin( np.abs( zlev-6000.))
z7=np.argmin( np.abs( zlev-7000.))
z10=np.argmin( np.abs( zlev-10000.))
z11=np.argmin( np.abs( zlev-11000.))
z12=np.argmin( np.abs( zlev-12000.))
z15=np.argmin( np.abs( zlev-15000.))
print(Eco.epwp_4D.shape)

#xv=Eco.zeta_4D[:,1,z6,4,:].mean(axis=1)
#xv=np.mean(Eco.zeta_4D[:,:,z10,4,:],axis=(1,2))
#xv=np.mean(Eco.zeta_4D[:,:,z7,:,:],axis=(1,2,3))
#xv=np.mean( Eco.tilt_4D[:,:,z7,:,:],axis=(1,2,3)  ) 

xvs=[]
xvs.append( np.mean( Eco.zeta_4D[:,:,z10:z3,:,:],axis=(1,2,3,4)  ) )
xvs.append( np.mean( Eco.tilt_4D[:,:,z10:z3,:,:],axis=(1,2,3,4)  ) )
xvs.append( np.mean( Eco.fgf_4D[:,:,z10:z3,:,:],axis=(1,2,3,4)  ) )
flds=['zeta','tilt','fgf']

yv=np.mean( Eco.epwp_4D[:,:,z10,:,:],axis=(3,2,1) )
#yv=np.mean( Eco.epwp_4D[:,nt_v-1,z10,:,:],axis=(2,1) )
print(yv.shape)

fig,axs=plt.subplots( 1, len(xvs), figsize=( (len(xvs)*8, 4 ) ) )
ip=0
for xv in xvs:
    ax=axs[ip]
    ax.scatter( xv,yv )
    r, p = stats.pearsonr(xv, yv)
    print(f"Patch mean {flds[ip]} vs patch mean epwp: r={r:.3f}, p={p:.2e}")
    ip=ip+1


In [ ]:
evoo=event_list[120]
fig,ax=plt.subplots( figsize=(20,8) )
ax.contourf(lon,lat,np.log(A.rho_epwp[120,z15,:,:]), levels=51, cmap='coolwarm')
for ev in evoo:
    ax.scatter( lon[ev['ix']] , lat[ev['iy']], marker='+' , c='black')
ax.contour( lon, lat, A.htopo , levels=[0.1,1,10,100,200,1000] )

In [ ]:

import importlib                                                                                                                                                                                                                                                                                              
import mlp_utils as mlu  
import predictors
importlib.reload(mlu) 
importlib.reload(RF)
importlib.reload(predictors)

#zlev=Eco.zlev.values

nv,nt_v,nz_v,ny_v,nx_v = np.shape( Eco.zeta_4D )

use_predictors= ['tilt_4D', 'zeta_4D']
trange=[nt_v-1, nt_v ] # No memory"

predictors,predictor_names,use_predictors,key_z = predictors.set_A_MM_genl( Eco=Eco, zlev=zlev, 
                                                                          trange=trange,
                                                                          use_predictors= use_predictors, 
                                                                          use_MM_winds=True)
#predictors,predictor_names,use_predictors,key_z = predictors.small_MM_set( Eco=Eco, zlev=zlev )


targ_scaling=1.
yv=targ_scaling*np.mean( Eco.epwp_4D[:,-1,z12,:,:],axis=(2,1) )

print( f"Predictors = {use_predictors}" )
print( f"key_z levels (m) = {[int(zlev[z]) for z in key_z]}" )
print( f"n_predictors = {len(predictors)}" )
print(f"target scaled by {targ_scaling}")
print( f"Events in {Eco.case},latlon={Eco.lon_range}X{Eco.lat_range}, exclude orography={Eco.exclude_orography} " )
print( f"Peak footprint={Eco.peak_footprint}" )
#print( f"Dates {A.start_date} to {A.end_date}, nsteps={A.nsteps}, stepsize={A.step_size} hrs" )
print( '\n' )

mlp_model, mlp_results = mlu.fit_mlp_general(
    predictors       = predictors,        # same list already built above
    predictor_names  = predictor_names,
    target           = yv,
    event_times      = Eco.time4D,
    train_interval   = (48, 248),         # same split as the RF
    test_interval    = (0, 28),
    dropout=0.1,
    hidden_dims      = (256, 256, 256, 128),
    weight_decay     = 1e-5,
    patience         = 100,
    lr_patience      = 3,
    batch_size=512,
    loss_power = 5,
    log_predictor_patterns = ['tilt','U_wv_src_mm',],)   # <-- new )

mlu.plot_distributions(mlp_results)   # <-- the new diagnostic  




"""


# fit
rf, results = RF.fit_rf_general(predictors=predictors, 
                                   predictor_names=predictor_names, 
                                   target=yv,
                                   event_times = Eco.time4D ,
                                   train_interval   = (48,248),
                                   test_interval    = (0,28),
                                   min_samples_leaf=10,
                                    )

RF.plot_rf_results(results, top_n=20)
"""


In [ ]:
print( f"{predictor_names}")

In [ ]:
n = 4
predlist = '\n'.join(',  '.join(predictor_names[i:i+n]) for i in range(0, len(predictor_names), n))
long_desc=short_desc + '\n' + '\n' + predlist


In [ ]:
print(poopypants)

In [ ]:
A.keys()

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(15,8) )

ax=axs[0]
co=ax.contourf(A.lon,A.lat,np.log(np.mean( A.epwp[:,z15,:,:],axis=0)) )
#co=ax.contourf(A.lon,A.lat,np.log10(np.mean( A.tilt[:,z3,:,:],axis=0)+1.e-8) )
ax.contour(A.lon,A.lat, A.htopo )
plt.colorbar(co)

ax=axs[1]
#plt.contourf(A.lon,A.lat,np.log(np.mean( A.epwp[:,z15,:,:],axis=0)) )
co=ax.contourf(A.lon,A.lat,np.log10(np.mean( A.tilt[:,z10:z3,:,:],axis=(0,1) )+1.e-8) ,levels=21 )
ax.contour(A.lon,A.lat, A.htopo )
plt.colorbar(co)


In [ ]:
Pi  = Co.pi()
rlat = (Pi/180.)*lat
coslat = np.cos( rlat )

coslat_jn = np.roll(coslat, -1)
coslat_js = np.roll(coslat, 1)


coslat_jn[-1] = coslat[-1]  # repeat last row
coslat_js[0] = coslat[0]  # repeat last row


plt.plot( coslat )
plt.plot( coslat_jn )
plt.plot( coslat_js )



In [ ]:
boo=np.log10(np.mean( A.tilt[:,z10:z3,:,:],axis=(0,1) )+1.e-8) 
boo=np.mean( A.u[:,z7:z7+1,:,:],axis=(0,1)  )
#plt.plot( boo[:,30] )
plt.plot( A.lat )
